In [ ]:
# Slightly more realistic problem formulation
# Optimise inventory same as 2.0_mpc.ipynb but add constraint that deliveries only come once 

# MPC forecsts: forecasts for same length as optimisation horizon
# Greedy forecasts: forecasts from same model but only for next day


In [1]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Mapping, Sequence
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import pulp

In [ ]:
# Constants

INPUT_BASE_PATH = Path("../data")
FORECASTS_PATH = Path("../results/ets/submissions")

# ==================================================
# OPTIMIZATION CONSTANTS / PARAMETERS / CONSTRAINTS
# ==================================================
TOTAL_STORAGE_CAPACITY = 35_000

# Gamma distributed
FOOD_VOLUME_DIST_PARAMS = {"shape": 5, "scale": 10}
HOUSEHOLD_VOLUME_DIST_PARAMS = {"shape": 10, "scale": 15}
HOBBIES_VOLUME_DIST_PARAMS = {"shape": 15, "scale": 17}

# Uniformly distributed
FOOD_STORAGE_COST_DIST_PARAMS = {"low": 0.1, "high": 0.5}
HOUSEHOLD_STORAGE_COST_DIST_PARAMS = {"low": 0.2, "high": 0.3}
HOBBIES_STORAGE_COST_DIST_PARAMS = {"low": 0.3, "high": 0.7}

FOODS_PRODUCT_IDS = [
    "FOODS_3_555_TX_2",
    "FOODS_3_376_TX_2",
    "FOODS_3_811_CA_2",
    "FOODS_1_218_TX_1",
    "FOODS_3_226_WI_3",
    "FOODS_3_070_WI_2",
    "FOODS_3_007_TX_2",
    "FOODS_3_444_TX_1",
    "FOODS_2_398_WI_3",
    "FOODS_3_540_WI_1",
]
HOUSEHOLD_PRODUCT_IDS = [
    "HOUSEHOLD_1_334_TX_1",
    "HOUSEHOLD_1_459_CA_2",
    "HOUSEHOLD_2_342_WI_2",
    "HOUSEHOLD_1_465_TX_3",
    "HOUSEHOLD_1_294_WI_2",
    "HOUSEHOLD_2_176_CA_3",
    "HOUSEHOLD_1_334_TX_2",
    "HOUSEHOLD_1_106_WI_2",
    "HOUSEHOLD_1_474_TX_2",
    "HOUSEHOLD_1_106_TX_1",
]
HOBBIES_PRODUCT_IDS = [
    "HOBBIES_1_048_WI_1",
    "HOBBIES_1_067_CA_3",
    "HOBBIES_1_158_TX_3",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_234_CA_3",
    "HOBBIES_1_254_CA_3",
    "HOBBIES_1_019_WI_1",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]
ALL_PRODUCT_IDS = list(FOODS_PRODUCT_IDS + HOUSEHOLD_PRODUCT_IDS + HOBBIES_PRODUCT_IDS)

# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1913
FORECAST_HORIZON = 7
MAX_FORECAST_TIMESTAMP = 1934
N_FORECAST_TIMESTAMPS = int(MAX_FORECAST_TIMESTAMP - MAX_TRAINING_TIMESTAMP)
FORECAST_TIMESTAMPS = range(MAX_TRAINING_TIMESTAMP, MAX_FORECAST_TIMESTAMP + 1)

RNG = np.random.default_rng(42)

#### RAW DATA

In [ ]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")

In [ ]:
# Unpivot SALES_DF to long df

SALES_DF = (
    SALES_TRAIN_EVALUATION
    .with_columns(id=pl.col("id").str.strip_suffix("_evaluation"))
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales"
    ).with_columns(d_index=pl.col("d").str.extract(r"^d_([0-9]+)").cast(pl.Int64))
)

SALES_DF.head()

#### SELL PRICES

In [ ]:
PRICES_DF = (
    SALES_DF
    .select(pl.col("id", "item_id", "store_id", "d", "d_index"))
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .join(
        other=CALENDAR_DATA.select(pl.col("wm_yr_wk"), pl.col("d")),
        how="left",
        on="d",
        validate="m:1",
    )
    .join(
        other=SELL_PRICES,
        on=["item_id", "store_id", "wm_yr_wk"],
        how="left",
    )
)

PRICES_DF.head()

#### STORAGE PRICES

In [ ]:
# Define storage cost per item as a proportion of average sell
# price for item.

average_sell_prices = (
    SELL_PRICES
    .with_columns(
        id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id")),
        dept_id=pl.col("item_id").str.extract(r"^([^_]+)")
    )
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .group_by(pl.col("id"), pl.col("dept_id"))
    .agg(avg_sell_price=pl.col("sell_price").mean())
    .sort(pl.col("dept_id"), pl.col("id"))
)

food_storage_costs_pct = RNG.uniform(**FOOD_STORAGE_COST_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_cost_pct = pl.DataFrame({"id": FOODS_PRODUCT_IDS, "cost_pct": food_storage_costs_pct})

household_storage_costs_pct = RNG.uniform(**HOUSEHOLD_STORAGE_COST_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_cost_pct = pl.DataFrame({"id": HOUSEHOLD_PRODUCT_IDS, "cost_pct": household_storage_costs_pct})

hobbies_storage_costs_pct = RNG.uniform(**HOBBIES_STORAGE_COST_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_cost_pct = pl.DataFrame({"id": HOBBIES_PRODUCT_IDS, "cost_pct": hobbies_storage_costs_pct})

COST_DF = (
    average_sell_prices.join(
        other=food_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_food"}),
        how="left",
        on="id",
    ).join(
        other=household_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_household"}),
        how="left",
        on="id"
    ).join(
        other=hobbies_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_hobbies"}),
        how="left",
        on="id"
    ).with_columns(
        cost_pct=pl.coalesce(pl.selectors.starts_with("cost_pct_"))
    ).with_columns(
        storage_cost=(pl.col("cost_pct") * pl.col("avg_sell_price")).round(2)
    ).drop(
        pl.selectors.starts_with("cost_pct"),
        pl.col("avg_sell_price"),
        pl.col("dept_id"),
    )
)

COST_DF

#### STORAGE VOLUME

In [ ]:
# Define storage volume per product

food_product_volumes = RNG.gamma(**FOOD_VOLUME_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_volume = pl.DataFrame({"id": FOODS_PRODUCT_IDS, "volume": food_product_volumes})

household_product_volumes = RNG.gamma(**HOUSEHOLD_VOLUME_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_volume = pl.DataFrame({"id": HOUSEHOLD_PRODUCT_IDS, "volume": household_product_volumes})

hobbies_product_volumes = RNG.gamma(**HOBBIES_VOLUME_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_volume = pl.DataFrame({"id": HOBBIES_PRODUCT_IDS, "volume": hobbies_product_volumes})

VOLUME_DF = (
    SALES_DF
    .select(["id"])
    .unique()
    .join(
        other=food_product_id_to_volume.rename({"volume": "volume_foods"}),
        on="id",
        how="left"
    ).join(
        other=household_product_id_to_volume.rename({"volume": "volume_household"}),
        on="id",
        how="left",
    ).join(
        other=hobbies_product_id_to_volume.rename({"volume": "volume_hobbies"}),
        on="id",
        how="left"
    )
    .with_columns(
        storage_volume=pl.coalesce(pl.selectors.starts_with("volume_"))
    ).drop(
        pl.selectors.starts_with("volume_"),
    )
)

VOLUME_DF

#### SALES FORECASTS

In [ ]:
# Baseline forecasts

class M5HistoricalAverageModel:
    def __init__(self, lookback: int = 1, horizon: int = 1):
        self.lookback = lookback
        self.horizon = horizon

        self._product_ids: tuple[str, ...] | None = None
        self._max_train_index_by_product: dict[str, int] | None = None

    @property
    def product_ids(self) -> tuple[str, ...] | None:
        return self._product_ids

    def fit(self, train_df: pl.DataFrame) -> "M5HistoricalAverageModel":
        # Get unique product ids
        self._product_ids = tuple(train_df["id"].unique())

        # Calculate min/max timestamps for each product.
        start_end_index = (
            train_df
            .group_by(["id"])
            .agg(t_end=pl.col("d_index").max())
            .with_columns(t_start=pl.col("t_end") - self.lookback)
        )
        self._max_train_index_by_product = {
            d["id"]: d["t_end"] 
            for d in start_end_index[["id", "t_end"]].to_dicts()
        }

        # Join back onto train_df, filter to index within range
        # and calculate average
        avg_sales = (
            train_df
            .join(start_end_index, on="id", how="inner")
            .filter(pl.col("d_index").is_between(pl.col("t_start"), pl.col("t_end"), closed="both"))
            .group_by(["id"])
            .agg(avg_sales=pl.col("sales").mean())
        )
        self._avg_sales_by_product = {
            d["id"]: d["avg_sales"] 
            for d in avg_sales[["id", "avg_sales"]].to_dicts()
        }
        
        return self

    def predict(self, horizon: int | None = None) -> pl.DataFrame:
        if horizon is None:
            horizon = self.horizon
        elif isinstance(horizon, int) and horizon < 1:
            raise ValueError(f"Horizon has to be > 1. Got {horizon=}")
        
        forecast_dfs: list[pl.DataFrame] = []
        for product_id in self.product_ids:
            max_t_for_product = self._max_train_index_by_product[product_id]
            avg_sales_for_product = self._avg_sales_by_product[product_id]
            product_forecast_df = pl.DataFrame(
                {
                    "id": [product_id for _ in range(horizon)],
                    "F_index": [max_t_for_product + i for i in range(1, horizon + 1)],
                    "sales": [avg_sales_for_product for _ in range(horizon)]
                }
            )
            forecast_dfs.append(product_forecast_df)

        return pl.concat(forecast_dfs)



### INVENTORY SOLVER

In [ ]:
@dataclass
class State:
    initial_stock: Mapping[str, int]


@dataclass
class ExogenousProblemData:
    demand_forecasts: Mapping[str, Sequence[float]]
    sell_prices: Mapping[str, Sequence[float]]
    storage_costs: Mapping[str, Sequence[float]]
    storage_volumes: Mapping[str, float]
    storage_capacity: float


@dataclass
class OptimalInventory:
    value_objective: float
    costs: Mapping[str, Sequence[float]]
    stock: Mapping[str, Sequence[int]]
    stock_additions: Mapping[str, Sequence[int]]
    sales: Mapping[str, Sequence[int]]
    


def optimise_inventory_greedy(
    product_ids: Sequence[str],
    state: State,
    data: ExogenousProblemData,
) -> OptimalInventory:
    problem = pulp.LpProblem(name="GreedyInventorySolver", sense=pulp.Lp.Minimize)

    # Decision variables
    sales_additions = pulp.LpVariable.dicts(name="stock_additions", indices=product_ids, lowBound=0, cat=pulp.LpInteger)
    stock = pulp.LpVariable.dicts(name="stock", indices=product_ids, lowBound=0, cat=pulp.LpInteger)
    cost_aux = pulp.LpVariable.dicts(name="cost_aux", indices=product_ids, lowBound=0, cat=pulp.LpContinuous)
    sales_aux = pulp.LpVariable.dicts(name="sales_aux", indices=product_ids, lowBound=0, cat=pulp.LpInteger)


def optimise_inventory_mpc(
    product_ids: Sequence[str],
    state: State,
    data: ExogenousProblemData,
    horizon: int = FORECAST_HORIZON,
    gamma: float = 0.95,
) -> OptimalInventory:
    ...


In [ ]:
product_ids = [
    "FOODS_3_376_TX_2",
    "FOODS_2_398_WI_3",
    "HOUSEHOLD_1_106_TX_1",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]

decision_timestamps = list(range(MAX_TRAINING_TIMESTAMP, MAX_FORECAST_TIMESTAMP + 1))
for decision_timestamp in decision_timestamps:
    break

initial_inventory: dict[str, int] = {}
for p in product_ids:
    initial_inventory[p] = 2
initial_state = State(initial_stock=initial_inventory)

storage_volumes = dict(VOLUME_DF.rows())
storage_costs = dict(COST_DF.rows())



In [52]:
# Greedy solver

product_ids = ["product1", "product2"]
initial_inventory = {"product1": 2, "product2": 3}
storage_volumes = {"product1": 0.75, "product2": 1.75}
storage_costs = {"product1": 1.5, "product2": 3.5}
sell_prices = {"product1": 1.25, "product2": 4.5}
demand = {"product1": 5, "product2": 10}

storage_capacity = 100

problem = pulp.LpProblem(name="GreedyInventorySolver", sense=pulp.LpMinimize)

# Decision variables
stock_addition_product_1 = pulp.LpVariable(name="product1", lowBound=0, cat=pulp.LpInteger)
stock_addition_product_2 = pulp.LpVariable(name="product2", lowBound=0, cat=pulp.LpInteger)

cost_aux_product_1 = pulp.LpVariable(name="cost_aux_product1", lowBound=0, cat=pulp.LpContinuous)
cost_aux_product_2 = pulp.LpVariable(name="cost_aux_product2", lowBound=0, cat=pulp.LpContinuous)

# Objective
problem += cost_aux_product_1 + cost_aux_product_2

# Constraints

# 1. Cost of missed sales = sell_price * max(0, demand - available stock).
# Available stock is initial + additions, so additions must be subtracted.
# cost_aux >= 0 already cuts off the negative (overstock) region of this line.
problem += cost_aux_product_1 >= sell_prices["product1"] * (demand["product1"] - (initial_inventory["product1"] + stock_addition_product_1))
problem += cost_aux_product_2 >= sell_prices["product2"] * (demand["product2"] - (initial_inventory["product2"] + stock_addition_product_2))

# 2. Cost of over stocking
problem += cost_aux_product_1 >= storage_costs["product1"] * (initial_inventory["product1"] + stock_addition_product_1 - demand["product1"])
problem += cost_aux_product_2 >= storage_costs["product2"] * (initial_inventory["product2"] + stock_addition_product_2 - demand["product2"])

# 3. Warehouse capacity
problem += (
    (initial_inventory["product1"] + stock_addition_product_1) * storage_volumes["product1"]
    + (initial_inventory["product2"] + stock_addition_product_2) * storage_volumes["product2"]
) <= storage_capacity

problem.solve(pulp.PULP_CBC_CMD(msg=False))
status = pulp.LpStatus[problem.status]
status

'Optimal'

In [53]:
stock_addition_product_1.value(), stock_addition_product_2.value()

(3.0, 7.0)

In [48]:
cost_aux_product_1.value(), cost_aux_product_2.value()

(3.75, 31.5)

In [49]:
missed_sales_p1 = sell_prices["product1"] * (demand["product1"] - initial_inventory["product1"] - stock_addition_product_1.value())
overstock_p1 = storage_costs["product1"] * (initial_inventory["product1"] + stock_addition_product_1.value() - demand["product1"])

missed_sales_p1, overstock_p1

(3.75, -4.5)

In [43]:
missed_sales_p2 = sell_prices["product2"] * (demand["product2"] - initial_inventory["product2"] - stock_addition_product_2.value())
overstock_p2 = storage_costs["product2"] * (initial_inventory["product2"] + stock_addition_product_2.value() - demand["product2"])

missed_sales_p2, overstock_p2

(31.5, -24.5)

In [33]:
# Initial storage volumes
(
    initial_inventory["product1"] * storage_volumes["product1"]
    + initial_inventory["product2"] * storage_volumes["product2"]
)

6.75

In [50]:
# Storage volume constraint.
(
    (initial_inventory["product1"] + stock_addition_product_1.value()) * storage_volumes["product1"]
    + (initial_inventory["product2"] + stock_addition_product_2.value()) * storage_volumes["product2"]
)

6.75